# 02. Grouping keys

**Tier:** Deep. **Why you'd use this:** Words that use the same letters share a key: the letters in sorted order. A dict of lists appends each word under that key.

**After this lesson you can:** build a key that ignores letter order; group items that share that key.
About 20 minutes. **Uses:** [First unique](01-first-unique-lesson.ipynb)


## Quick review

1. What makes two strings anagrams of each other?
2. Why can a sorted string work as a grouping key?

<details><summary>Answers</summary>

For 02 anagram lesson: skim [First unique](01-first-unique-lesson.ipynb) if needed.
</details>


## Recognize it

You need to put items into buckets where membership depends on a derived key, not on the item's original form. The prompt says "group by", "same letters", "same remainder", or "items that match under a transform".

Signals: many items, one shared property after a transform, and you want the groups (or a yes/no that two items share a key). A look-alike that is not this pattern: sorting the whole list once when you only need order — that is a sort, not a grouping dict.

Another look-alike: counting how often each key appears without keeping the members. That is a `Counter` or a frequency map. You still derive a key, but the answer is a number per key, not the list of items that share it. Reach for a grouping dict when you will later walk the members inside each bucket.


## Why it works this way

Grouping without a key means comparing every item to every other item. For `n` words that is roughly `n²` equality checks, and each check may scan the whole word. A dict turns the key into the address of a bucket. You pay to build the key once per item, then append into that bucket in average constant time. See [Dicts](../../internals/04-dicts/01-hash-table-lesson.ipynb) for how hashing finds the bucket.

The key must be hashable and stable. A list of letters is not a valid dict key because it can change; a tuple of the same letters is fine. For anagram-style grouping the sorted letters are a canonical form: `eat` and `tea` both become `('a', 'e', 't')`. For the sibling here the key is simpler — `value % 3` — but the dict shape is the same: key → list of members.

`collections.defaultdict(list)` removes the "create the list on first sight" branch. Arrival order inside a bucket is the order you append, which is usually what you want. Converting to a plain `dict` at the end is optional; it only matters if a caller rejects the defaultdict type.

Cost follows the key work. Building `value % 3` is constant. Building `tuple(sorted(word))` costs roughly the length of the word times a log factor for the sort. Memory holds every item once across all buckets, plus the keys. Prefer this when you will ask "who shares this key?" many times. Prefer sorting the full list when the only goal is global order and you never need random access by key.

The design reason for a canonical key is uniqueness under the transform you care about. Two different remainders must never collide into one bucket, and two values with the same remainder must never land apart. That is why `%` is the right operator here and why anagram keys sort letters instead of preserving the original spelling. Once the key is honest, the dict does the rest of the grouping work for free.


## When to use it

Use a grouping dict when items that share a derived key should land in the same bucket, and you need those buckets afterward. Use a Counter when you only need frequencies, not the members. Use a set when membership is enough and groups do not matter. Skip the dict when `n` is tiny and a nested compare is clearer.

If the prompt only asks whether two strings are anagrams of each other, a pair of Counters (or one Counter and a subtract) is enough — you do not need to group a whole list. Reach for the dict-of-lists shape when many items must be partitioned, not when a single yes/no check is the whole problem.


## Brute force, and why it's slow

Pair every value with every later value that shares the same remainder. Count how many comparisons that takes.


In [1]:
def by_remainder_slow(nums):
    groups = {0: [], 1: [], 2: []}
    comparisons = 0
    placed = [False] * len(nums)
    for index, value in enumerate(nums):
        if placed[index]:
            continue
        bucket = [value]
        placed[index] = True
        for other in range(index + 1, len(nums)):
            comparisons += 1
            if nums[other] % 3 == value % 3:
                bucket.append(nums[other])
                placed[other] = True
        groups[value % 3] = bucket
    return groups, comparisons


print(by_remainder_slow([1, 3, 4, 6, 8, 9]))


({0: [3, 6, 9], 1: [1, 4], 2: [8]}, 10)


## Key insight

The remainder `value % 3` is the bucket key. Append each value under that key as you scan once — no pairwise compare. Same key means same bucket; different keys never merge.


## Worked example

Group numbers by their remainder when divided by 3. Same dict-of-lists shape as grouping words by a letter key.


In [2]:
from collections import defaultdict


def by_remainder(nums: list[int]) -> dict[int, list[int]]:
    # 1. The key is the remainder.
    groups = defaultdict(list)
    for value in nums:
        # 2. Append in arrival order.
        groups[value % 3].append(value)
    return dict(groups)


print(by_remainder([1, 3, 4, 6]))
print(by_remainder([2, 5, 8]))


{1: [1, 4], 0: [3, 6]}
{2: [2, 5, 8]}


## Trace

`[1, 3, 4, 6]`:

| value | key (`% 3`) | groups after step |
|---|---|---|
| 1 | 1 | `{1: [1]}` |
| 3 | 0 | `{1: [1], 0: [3]}` |
| 4 | 1 | `{1: [1, 4], 0: [3]}` |
| 6 | 0 | `{1: [1, 4], 0: [3, 6]}` |


## Cost

One pass: time grows with `n`. Each value is stored once, so memory grows with `n`. The slow pairing version rechecks remainders across later indexes and grows like `n²` comparisons on distinct-remainder-heavy inputs. Key construction stays constant for modulo; for anagram keys it grows with word length.


## Variants

Not solved here: [Group Anagrams](../../challenges/problems/02-group-anagrams.ipynb), [Top K Frequent Elements](../../challenges/problems/15-top-k-frequent-elements.ipynb), [Two Sum](../../challenges/problems/01-two-sum.ipynb).


## Common mistake

This bug is silent. The key uses integer division instead of remainder, so `1` and `4` land in different buckets even though both leave remainder `1`. There is no traceback. The print shows split groups, and the right answer keeps `1` and `4` together.


In [3]:

def by_remainder_wrong(nums):
    groups = defaultdict(list)
    for value in nums:
        groups[value // 3].append(value)  # wrong key
    return dict(groups)


print(by_remainder_wrong([1, 3, 4, 6]))


{0: [1], 1: [3, 4], 2: [6]}


The fix uses `%`, not `//`. Remainders `0` and `1` are the buckets. The worked example does that, so `[1, 3, 4, 6]` groups as `{1: [1, 4], 0: [3, 6]}`.


In [4]:
print(by_remainder([1, 3, 4, 6]))


{1: [1, 4], 0: [3, 6]}


## Python idioms

`defaultdict(list)` creates the bucket on first write. `Counter` compares letter bags in one expression when you only need a yes/no for two strings, not the groups.


In [5]:
from collections import Counter


def same_letters(left: str, right: str) -> bool:
    return Counter(left) == Counter(right)


print(same_letters("eat", "tea"))
print(same_letters("eat", "tan"))


True
False


## Recap

- Derive a hashable key, then append into that bucket.
- `%` is remainder; `//` is a different key.
- `defaultdict(list)` drops the "create on first sight" branch.

Next: [practice](02-anagram-practice.ipynb).
